In [ ]:
import sqlite3
con=sqlite3.connect("mutual_funds.db")
cursor=con.cursor()
cursor.execute("""
CREATE TABLE dim_fund(
fund_key INTEGER PRIMARY KEY AUTOINCREMENT,
amfi_code INTEGER UNIQUE NOT NULL,
scheme_name TEXT NOT NULL,
fund_house TEXT,
category TEXT,
plan TEXT,
risk_grade TEXT);
  """)
con.commit()
con.close()

OperationalError: table dim_fund already exists

In [ ]:
cursor.execute("""
CREATE TABLE  fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    fund_key INTEGER NOT NULL,
    date_key INTEGER NOT NULL,
    nav REAL NOT NULL,

    FOREIGN KEY (fund_key) REFERENCES dim_fund(fund_key),
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE  fact_transaction (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    fund_key INTEGER NOT NULL,
    date_key INTEGER NOT NULL,

    investor_id INTEGER,
    transaction_type TEXT,
    amount REAL,
    units REAL,

    FOREIGN KEY (fund_key) REFERENCES dim_fund(fund_key),
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE  fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,

    fund_key INTEGER NOT NULL,
    date_key INTEGER NOT NULL,

    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,

    FOREIGN KEY (fund_key) REFERENCES dim_fund(fund_key),
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
)
""")

In [ ]:
cursor.execute("""
CREATE TABLE  fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,

    fund_key INTEGER NOT NULL,
    date_key INTEGER NOT NULL,

    aum_crore REAL,

    FOREIGN KEY (fund_key) REFERENCES dim_fund(fund_key),
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key)
)
""")

In [ ]:
con.commit()

In [ ]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""")
for row in cursor.fetchall():
  print(row[0])

dim_fund
fact_aum
fact_nav
fact_performance
fact_transaction
sqlite_sequence


In [ ]:
import pandas as pd
from sqlalchemy import create_engine


In [ ]:
engine=create_engine("sqlite:///mutual_funds.db")

In [ ]:
fund_master = pd.read_csv("/content/drive/MyDrive/Google AI Studio/01_fund_master_clean.csv")

nav_history = pd.read_csv("/content/drive/MyDrive/Google AI Studio/nav_history_clean.csv")

transactions = pd.read_csv("/content/drive/MyDrive/Google AI Studio/08_investor_transactions.csv")

performance = pd.read_csv("/content/drive/MyDrive/Google AI Studio/07_scheme_performance_clean.csv")

In [ ]:
print(fund_master.shape)
print(nav_history.shape)
print(transactions.shape)
print(performance.shape)

(40, 15)
(45962, 3)
(32778, 13)
(40, 20)


In [ ]:
fund_master.to_sql("dim_fund",engine,if_exists="append",index=False)
nav_history.to_sql("fact_nav",engine,if_exists="append",index=False)
transactions.to_sql("fact_transaction",engine,if_exists="append",index=False)
performance.to_sql("fact_performance",engine,if_exists="append",index=False)

40

In [ ]:
from sqlalchemy import text

with engine.connect() as con:
    result = con.execute(text("""
        SELECT name
        FROM sqlite_master
        WHERE type='table'
    """))

    for row in result:
        print(row)

('dim_fund',)
('fact_nav',)
('fact_transaction',)
('fact_performance',)


In [ ]:
from sqlalchemy import text

with engine.connect() as con:
    result = con.execute(
        text("""
        SELECT *
        FROM dim_fund
        WHERE amfi_code BETWEEN 100000 AND 120000
        """)
    )

    for row in result:
        print(row)

(119551, 'SBI Mutual Fund', 'SBI Bluechip Fund - Regular Plan - Growth', 'Equity', 'Large Cap', 'Regular', '2006-02-14', 'NIFTY 100 TRI', 1.54, 1.0, 500, 1000, 'Sohini Andani', 'Moderate', 'EC01')
(119552, 'SBI Mutual Fund', 'SBI Bluechip Fund - Direct Plan - Growth', 'Equity', 'Large Cap', 'Direct', '2013-01-01', 'NIFTY 100 TRI', 0.66, 1.0, 500, 1000, 'Sohini Andani', 'Moderate', 'EC01')
(119598, 'SBI Mutual Fund', 'SBI Small Cap Fund - Regular Plan - Growth', 'Equity', 'Small Cap', 'Regular', '2009-09-09', 'BSE 250 SmallCap TRI', 1.43, 1.0, 500, 1000, 'R. Srinivasan', 'Very High', 'EC03')
(119599, 'SBI Mutual Fund', 'SBI Small Cap Fund - Direct Plan - Growth', 'Equity', 'Small Cap', 'Direct', '2013-01-01', 'BSE 250 SmallCap TRI', 0.72, 1.0, 500, 1000, 'R. Srinivasan', 'Very High', 'EC03')
(119120, 'SBI Mutual Fund', 'SBI Magnum Gilt Fund - Regular Plan - Growth', 'Debt', 'Gilt', 'Regular', '2000-12-30', 'CRISIL Dynamic Gilt Index', 0.77, 0.0, 500, 1000, 'Dinesh Ahuja', 'Low', 'DC02')

In [ ]:
from sqlalchemy import text

with engine.connect() as con:
    result = con.execute(text("""
        PRAGMA table_info(fact_performance)
    """))

    for row in result:
        print(row)

(0, 'amfi_code', 'BIGINT', 0, None, 0)
(1, 'scheme_name', 'FLOAT', 0, None, 0)
(2, 'fund_house', 'FLOAT', 0, None, 0)
(3, 'category', 'FLOAT', 0, None, 0)
(4, 'plan', 'FLOAT', 0, None, 0)
(5, 'return_1yr_pct', 'FLOAT', 0, None, 0)
(6, 'return_3yr_pct', 'FLOAT', 0, None, 0)
(7, 'return_5yr_pct', 'FLOAT', 0, None, 0)
(8, 'benchmark_3yr_pct', 'FLOAT', 0, None, 0)
(9, 'alpha', 'FLOAT', 0, None, 0)
(10, 'beta', 'FLOAT', 0, None, 0)
(11, 'sharpe_ratio', 'FLOAT', 0, None, 0)
(12, 'sortino_ratio', 'FLOAT', 0, None, 0)
(13, 'std_dev_ann_pct', 'FLOAT', 0, None, 0)
(14, 'max_drawdown_pct', 'FLOAT', 0, None, 0)
(15, 'aum_crore', 'BIGINT', 0, None, 0)
(16, 'expense_ratio_pct', 'FLOAT', 0, None, 0)
(17, 'morningstar_rating', 'BIGINT', 0, None, 0)
(18, 'risk_grade', 'FLOAT', 0, None, 0)
(19, 'anomaly_flag', 'BOOLEAN', 0, None, 0)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
   result=con.execute(text(""" SELECT count(*) FROM dim_fund """))
   print(result.fetchone())

with engine.connect() as con:
  result=con.execute(text(""" SELECT count(*) FROM fact_nav """))
  print(result.fetchone())

with engine.connect() as con:
  result=con.execute(text(""" SELECT count(*) FROM fact_transaction """))
  print(result.fetchone())

with engine.connect() as con:
  result=con.execute(text(""" SELECT count(*) FROM fact_performance """))
  print(result.fetchone())





(120,)
(137886,)
(98334,)
(120,)


In [ ]:
print(fund_master.columns.tolist())
print(nav_history.columns.tolist())
print(transactions.columns.tolist())
print(performance.columns.tolist())

['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
['amfi_code', 'date', 'nav']
['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'anomaly_flag']


In [ ]:
from sqlalchemy import text

with engine.connect() as con:
    result = con.execute(text("PRAGMA table_info(fact_nav)"))

    for row in result:
        print(row)

(0, 'amfi_code', 'BIGINT', 0, None, 0)
(1, 'date', 'TEXT', 0, None, 0)
(2, 'nav', 'FLOAT', 0, None, 0)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""PRAGMA table_info(fact_performance) """))
  for row in result:
    print(row)

(0, 'amfi_code', 'BIGINT', 0, None, 0)
(1, 'scheme_name', 'FLOAT', 0, None, 0)
(2, 'fund_house', 'FLOAT', 0, None, 0)
(3, 'category', 'FLOAT', 0, None, 0)
(4, 'plan', 'FLOAT', 0, None, 0)
(5, 'return_1yr_pct', 'FLOAT', 0, None, 0)
(6, 'return_3yr_pct', 'FLOAT', 0, None, 0)
(7, 'return_5yr_pct', 'FLOAT', 0, None, 0)
(8, 'benchmark_3yr_pct', 'FLOAT', 0, None, 0)
(9, 'alpha', 'FLOAT', 0, None, 0)
(10, 'beta', 'FLOAT', 0, None, 0)
(11, 'sharpe_ratio', 'FLOAT', 0, None, 0)
(12, 'sortino_ratio', 'FLOAT', 0, None, 0)
(13, 'std_dev_ann_pct', 'FLOAT', 0, None, 0)
(14, 'max_drawdown_pct', 'FLOAT', 0, None, 0)
(15, 'aum_crore', 'BIGINT', 0, None, 0)
(16, 'expense_ratio_pct', 'FLOAT', 0, None, 0)
(17, 'morningstar_rating', 'BIGINT', 0, None, 0)
(18, 'risk_grade', 'FLOAT', 0, None, 0)
(19, 'anomaly_flag', 'BOOLEAN', 0, None, 0)


In [ ]:
from sqlalchemy import text

with engine.connect() as con:
    result = con.execute(text("""
       SELECT d.scheme_name, p.aum_crore
      FROM fact_performance p
      JOIN dim_fund d
      ON p.amfi_code = d.amfi_code
       ORDER BY p.aum_crore DESC
       LIMIT 10;
    """))

    print(result.fetchone())

('Mirae Asset Emerging Bluechip Fund - Regular - Growth', 49046)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  SELECT strftime('%Y-%m',date) AS month,
   ROUND(AVG(nav),2)AS avg_nav
   FROM fact_nav
   GROUP BY month
   ORDER BY month"""))
  print(result.fetchone())

('2022-01', 207.26)


In [ ]:
with engine.connect() as con:
    result = con.execute(text("""
        PRAGMA table_info(fact_transaction)
    """))

    for row in result:
        print(row)

(0, 'investor_id', 'TEXT', 0, None, 0)
(1, 'transaction_date', 'TEXT', 0, None, 0)
(2, 'amfi_code', 'BIGINT', 0, None, 0)
(3, 'transaction_type', 'TEXT', 0, None, 0)
(4, 'amount_inr', 'BIGINT', 0, None, 0)
(5, 'state', 'TEXT', 0, None, 0)
(6, 'city', 'TEXT', 0, None, 0)
(7, 'city_tier', 'TEXT', 0, None, 0)
(8, 'age_group', 'TEXT', 0, None, 0)
(9, 'gender', 'TEXT', 0, None, 0)
(10, 'annual_income_lakh', 'FLOAT', 0, None, 0)
(11, 'payment_mode', 'TEXT', 0, None, 0)
(12, 'kyc_status', 'TEXT', 0, None, 0)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
 result=con.execute(text("""
  SELECT strftime('%Y',transaction_date) AS year,
  SUM(amount_inr) AS SIP
  FROM fact_transaction
  WHERE transaction_type='SIP'
  GROUP BY year
  ORDER BY year """))
 print(result.fetchone())

('2024', 459699156)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  SELECT state,COUNT(*) AS transaction_count
  FROM fact_transaction
  GROUP BY state
  ORDER BY transaction_count DESC"""))
  print(result.fetchone())

('Punjab', 8895)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  PRAGMA table_info(fact_performance)"""))
  for row in result:
    print(row)

(0, 'amfi_code', 'BIGINT', 0, None, 0)
(1, 'scheme_name', 'FLOAT', 0, None, 0)
(2, 'fund_house', 'FLOAT', 0, None, 0)
(3, 'category', 'FLOAT', 0, None, 0)
(4, 'plan', 'FLOAT', 0, None, 0)
(5, 'return_1yr_pct', 'FLOAT', 0, None, 0)
(6, 'return_3yr_pct', 'FLOAT', 0, None, 0)
(7, 'return_5yr_pct', 'FLOAT', 0, None, 0)
(8, 'benchmark_3yr_pct', 'FLOAT', 0, None, 0)
(9, 'alpha', 'FLOAT', 0, None, 0)
(10, 'beta', 'FLOAT', 0, None, 0)
(11, 'sharpe_ratio', 'FLOAT', 0, None, 0)
(12, 'sortino_ratio', 'FLOAT', 0, None, 0)
(13, 'std_dev_ann_pct', 'FLOAT', 0, None, 0)
(14, 'max_drawdown_pct', 'FLOAT', 0, None, 0)
(15, 'aum_crore', 'BIGINT', 0, None, 0)
(16, 'expense_ratio_pct', 'FLOAT', 0, None, 0)
(17, 'morningstar_rating', 'BIGINT', 0, None, 0)
(18, 'risk_grade', 'FLOAT', 0, None, 0)
(19, 'anomaly_flag', 'BOOLEAN', 0, None, 0)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
   SELECT d.scheme_name,p.expense_ratio_pct
   FROM fact_performance p
   JOIN dim_fund d
   ON p.amfi_code=d.amfi_code
   WHERE p.expense_ratio_pct<1
   ORDER BY p.expense_ratio_pct"""))
  print(result.fetchone())

('Nippon India Gilt Securities Fund - Regular - Growth', 0.55)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  SELECT d.scheme_name,
       p.return_5yr_pct
FROM fact_performance p
JOIN dim_fund d
ON p.amfi_code = d.amfi_code
ORDER BY p.return_5yr_pct DESC
LIMIT 10;"""))
  print(result.fetchone())

('ABSL Small Cap Fund - Regular - Growth', 23.8)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  SELECT d.category,
       ROUND(AVG(p.return_3yr_pct),2) AS avg_return
FROM fact_performance p
JOIN dim_fund d
ON p.amfi_code = d.amfi_code
GROUP BY d.category
ORDER BY avg_return DESC;"""))
  print(result.fetchone())

('Equity', 15.47)


In [ ]:
from sqlalchemy import text
with engine.connect() as con:
  result=con.execute(text("""
  SELECT strftime('%Y-%m', transaction_date) AS month,
       COUNT(*) AS total_transactions
FROM fact_transaction
GROUP BY month
ORDER BY month;"""))
  print(result.fetchone())

('2024-01', 5847)
